# Preprocessing Pipeline (Production-Ready)

This notebook creates reproducible ILPD preprocessing artifacts from raw data and saves outputs under the repository `data/` directory.

## Outputs
- `data/interim/ILPD_cleaned.csv`
- `produced_reports/docs/ILPD_outlier_report.csv`
- `data/processed/ILPD_clinically_capped.csv`
- `data/processed/ILPD_robust_scaled_features.csv`
- `produced_reports/docs/ILPD_preprocessing_metadata.json`


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler


In [ ]:
def resolve_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory")

ROOT = resolve_repo_root(Path.cwd())
RAW_PATH = ROOT / "data" / "raw" / "ILPD.csv"
INTERIM_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "produced_reports" / "docs"

for directory in [INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")
print(f"Raw dataset: {RAW_PATH}")
print(f"Interim output dir: {INTERIM_DIR}")
print(f"Processed output dir: {PROCESSED_DIR}")
print(f"Reports output dir: {REPORTS_DIR}")


## 1) Load and standardize schema

In [ ]:
COLUMN_NAMES = [
    "Age",
    "Gender",
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Total_Proteins",
    "Albumin",
    "Albumin_and_Globulin_Ratio",
    "Result",
]

df = pd.read_csv(RAW_PATH, header=None, names=COLUMN_NAMES)
print(df.shape)
df.head()


## 2) Missing/invalid/infinite checks + duplicate removal + gender encoding

This step performs data quality checks first, encodes `Gender`, and removes duplicate rows before missing-value treatment.

In [ ]:
missing_before = df.isnull().sum().to_dict()

# Invalid/infinite checks BEFORE missing-value treatment
invalid_checks = {
    "Age_negative": int((df["Age"] < 0).sum()),
    "Age_too_high": int((df["Age"] > 120).sum()),
    "Bilirubin_negative": int((df["Total_Bilirubin"] < 0).sum()),
    "Proteins_negative": int((df["Total_Proteins"] < 0).sum()),
    "Albumin_negative": int((df["Albumin"] < 0).sum()),
    "AG_ratio_negative": int((df["Albumin_and_Globulin_Ratio"] < 0).sum()),
    "Infinite_numeric_values": int(np.isinf(df.select_dtypes(include=[np.number])).sum().sum()),
}

# Encode categorical column for modeling compatibility
df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})

# Remove exact duplicates
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

print("Missing (before):", missing_before)
print("Invalid checks:", invalid_checks)
print("Duplicates removed:", duplicates_removed)
print("Shape after dedup:", df.shape)

## 3) Missing value treatment with group-wise median

Single selected strategy for `Albumin_and_Globulin_Ratio`:
- group-wise median by `Gender`
- global median fallback

In [ ]:
impute_col = "Albumin_and_Globulin_Ratio"
imputation_strategy = "group_median_by_gender_with_global_fallback"

global_median = df[impute_col].median()
df[impute_col] = df.groupby("Gender")[impute_col].transform(lambda s: s.fillna(s.median()))
df[impute_col] = df[impute_col].fillna(global_median)

missing_after = df.isnull().sum().to_dict()

print("Imputation strategy:", imputation_strategy)
print("Missing (after):", missing_after)
print("Shape after imputation:", df.shape)

## 4) Outlier assessment report (IQR, Z-score, Modified Z-score)

This creates a report only; it does not drop rows.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.drop(["Result", "Gender"])

report_rows = []

for col in num_cols:
    series = df[col]

    # IQR
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    iqr_lower = q1 - 1.5 * iqr
    iqr_upper = q3 + 1.5 * iqr
    iqr_outliers = int(((series < iqr_lower) | (series > iqr_upper)).sum())

    # Z-score
    std = series.std(ddof=0)
    mean = series.mean()
    if std == 0:
        z_outliers = 0
    else:
        z = ((series - mean) / std).abs()
        z_outliers = int((z > 3).sum())

    # Modified Z-score
    median = series.median()
    mad = np.median(np.abs(series - median))
    if mad == 0:
        mz_outliers = 0
    else:
        modified_z = (0.6745 * (series - median) / mad).abs()
        mz_outliers = int((modified_z > 3.5).sum())

    report_rows.append({
        "feature": col,
        "rows": int(len(df)),
        "iqr_outliers": iqr_outliers,
        "iqr_pct": round(100 * iqr_outliers / len(df), 2),
        "zscore_outliers": z_outliers,
        "zscore_pct": round(100 * z_outliers / len(df), 2),
        "modified_z_outliers": mz_outliers,
        "modified_z_pct": round(100 * mz_outliers / len(df), 2),
    })

outlier_report = pd.DataFrame(report_rows).sort_values("iqr_pct", ascending=False)
outlier_report.head(10)


## 5) Clinically bounded capping (winsorization-style)

Caps only biologically implausible extremes while preserving records.

In [ ]:
clinical_limits = {
    "Age": (0, 100),
    "Total_Bilirubin": (0, 50),
    "Direct_Bilirubin": (0, 30),
    "Alkaline_Phosphotase": (0, 2000),
    "Alamine_Aminotransferase": (0, 5000),
    "Aspartate_Aminotransferase": (0, 10000),
    "Total_Proteins": (0, 15),
    "Albumin": (0, 6.0),
    "Albumin_and_Globulin_Ratio": (0, 3.0),
}

capped_df = df.copy()
for col, (min_val, max_val) in clinical_limits.items():
    if col in capped_df.columns:
        capped_df[col] = capped_df[col].clip(lower=min_val, upper=max_val)

capped_df.head()


## 6) Robust-scaled feature dataset (for modeling convenience)

`Result` is retained separately and not scaled.

In [ ]:
feature_cols = [c for c in capped_df.columns if c != "Result"]

scaler = RobustScaler()
scaled_features = pd.DataFrame(
    scaler.fit_transform(capped_df[feature_cols]),
    columns=feature_cols,
)

scaled_with_target = scaled_features.copy()
scaled_with_target["Result"] = capped_df["Result"].values

scaled_with_target.head()


## 7) Persist pipeline artifacts to `data/`

In [ ]:
clean_path = INTERIM_DIR / "ILPD_cleaned.csv"
outlier_path = REPORTS_DIR / "ILPD_outlier_report.csv"
capped_path = PROCESSED_DIR / "ILPD_clinically_capped.csv"
scaled_path = PROCESSED_DIR / "ILPD_robust_scaled_features.csv"
metadata_path = REPORTS_DIR / "ILPD_preprocessing_metadata.json"

df.to_csv(clean_path, index=False)
outlier_report.to_csv(outlier_path, index=False)
capped_df.to_csv(capped_path, index=False)
scaled_with_target.to_csv(scaled_path, index=False)

metadata = {
    "input": str(RAW_PATH),
    "rows_raw": int(pd.read_csv(RAW_PATH, header=None).shape[0]),
    "rows_cleaned": int(df.shape[0]),
    "columns": list(df.columns),
    "missing_before": missing_before,
    "missing_after": missing_after,
    "imputation_strategy": imputation_strategy,
    "invalid_checks": invalid_checks,
    "duplicates_removed": duplicates_removed,
    "outputs": {
        "cleaned": str(clean_path),
        "outlier_report": str(outlier_path),
        "clinically_capped": str(capped_path),
        "robust_scaled_features": str(scaled_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2))

print("Saved:")
print(clean_path)
print(outlier_path)
print(capped_path)
print(scaled_path)
print(metadata_path)
